##### Copyright 2019 The TensorFlow Authors.

# Deep Learning · 11. gyakorlat

## 2. Szöveggenerálás GRU-val

A kiindulópont a [TensorFlow szöveggenerálási oktatóanyaga](https://www.tensorflow.org/text/tutorials/text_generation). Ehhez a gyakorlathoz ezt a kiosztott példányt használd. A korpusz és a generált szöveg magyar.

### T00 — TensorFlow, Keras és GPU ellenőrzése

**Első futtatható cella.** Colabban válaszd a GPU-t és a **2026.04** futtatókörnyezet-verziót, mielőtt bármit futtatsz. Ez a technikai előkészítés állítja be a régi Keras-felületet, amelyre az eredeti modell épül.

In [ ]:
# T00 — Colab setup for the original legacy Keras model.
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    tf_version = version("tensorflow")
except PackageNotFoundError:
    tf_version = ""
if not tf_version.startswith("2.19."):
    raise RuntimeError("Colab: choose GPU and runtime version 2026.04 (TensorFlow 2.19), then rerun T00.")

try:
    legacy_version = version("tf-keras")
except PackageNotFoundError:
    legacy_version = None

if legacy_version != "2.19.0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "tf-keras==2.19.0"
    ])

os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf

keras_backend = tf.keras.__name__
gpu_devices = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("Keras:", keras_backend)
print("GPU:", gpu_devices)
if not keras_backend.startswith("tf_keras"):
    raise RuntimeError("Legacy Keras was not loaded. Restart the Colab session and run T00 first.")
if not gpu_devices:
    raise RuntimeError("Colab: Runtime > Change runtime type > GPU > Save. Then rerun T00.")


In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

A modell Rejtő Jenő *Piszkos Fred, a kapitány* című szövegéből a következő karaktert tanulja meg előre jelezni. Ha a megjósolt karaktert visszaadjuk neki bemenetként, új szöveget tud képezni.

Korábbi futásból származó mintaszöveg (szemléltetés).

<pre>
An saját magaszására társal lett van a pitánsággal, könnyedéseket.
A herceg szeme le régen is tűt. Mehes. Fülig Jimmy? Mi ajra, ha Hurcunk Fernántesz! És igen ötök boripán az találgassza, Felség, angol hibetkézik az én nevemben, azt hitte, hogy néhány óráj baj vas, hát de nem hites, előfordul egy erőtel sem mozdult. Pedig a hajón vérn A fiú uralkodásai és lenéssel az illetőkeá.
- Hány és GrAndszervez erted?
Így öreg Wilson Hutchins (az amerikai fűtő már csak azért sem vonállattag érezte, amelyen a pincér aztán ki történt. Fel tudum. Most már közzem (Övig nem ívás, hanem Jiment furcsa, akkor ja mokdanás, mert a brót és fél személyesen ismerem.
- Warins a vőlegverék el - mondja a stamát... Az is a pillanatra! - jegyezte meg közölte?
- Ezent lesz a borízsal állt. - Es osztálybal. Öngyen közt kenyeres tudja, ő a próféte megtoválját, így szólt:
- Integessen Felség, hogy egy nap alatt sok mindent tett, embere! Hozzt, hogy nem keresem tovább vezeteti Bannera. Tevissza kelé krabitány az á

</pre>

A tanítás során a modell helyesírási és központozási mintákat vehet fel. A felismerhető részletek mellett töredékek és hibás szavak is megjelenhetnek.

## Előkészítés

### T01 — TensorFlow és segédkönyvtárak

In [ ]:
import tensorflow as tf

import numpy as np
import os
import time

import os
os.environ['TF_USE_LEGACY_KERAS']='1'

A gyakorlat során a megadott Rejtő-korpuszt használjuk.

### T02 — A korpusz letöltése

In [ ]:
#path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')
path_to_file = tf.keras.utils.get_file('piszkosfred.txt', 'https://raw.githubusercontent.com/karsarobert/DeepLearning2026/main/piszkosfred.txt')

### T03 — UTF-8 szöveg beolvasása

In [ ]:
# Read, then decode for py2 compat.
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
# length of text is the number of characters in it
print(f'Length of text: {len(text)} characters')

### T04 — Az első 250 karakter

In [ ]:
# Take a look at the first 250 characters in text
print(text[:250])

### T05 — A karakterkészlet

In [ ]:
# The unique characters in the file
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

## A korpusztól a tanítópárokig

### T06 — Karakterből azonosító, azonosítóból karakter

In [ ]:
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text]) #szöveget pontosabban a karaktereket számokká konvertáljuk

### T07 — Az indexszótár megtekintése

In [ ]:
print('{')
for char,_ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

### T08 — Az első 13 karakter kódolása

In [ ]:
print ('{} ---- characters mapped to int ---- > {}'.format(repr(text[:13]), text_as_int[:13]))

Minden időlépésnél a következő karakter a tanítócél. A bemenet és a cél egyaránt indexsorozat.

Például egy `Helló` darab esetén a bemenet `Hell`, a cél `elló`.

### T09 — Karakterenkénti adatfolyam

In [ ]:
seq_length = 100
examples_per_epoch = len(text)//(seq_length+1)


char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

for i in char_dataset.take(10):
  print(idx2char[i.numpy()])

### T10 — 101 karakteres darabok

In [ ]:
sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

for item in sequences.take(5):
  print(repr(''.join(idx2char[item.numpy()])))

### T11 — Bemenet és cél szétválasztása

In [ ]:
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text



### T12 — Az eltolás kipróbálása

In [ ]:
split_input_target(list("Tensorflow"))

### T13 — Címkepárok a teljes adatfolyamon

In [ ]:
dataset = sequences.map(split_input_target)

### T14 — Egy bemeneti–cél pár megtekintése

In [ ]:
for input_example, target_example in  dataset.take(1):
  print ('Input data: ', repr(''.join(idx2char[input_example.numpy()])))
  print ('Target data:', repr(''.join(idx2char[target_example.numpy()])))

A tanítópárokat keverjük, majd 64-es batchbe rendezzük. A sorozat belső sorrendje megmarad.

### T15 — Öt időlépés párosítása

In [ ]:
for i, (input_idx, target_idx) in enumerate(zip(input_example[:5], target_example[:5])):
    print("Step {:4d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx])))

### T16 — Keverés és 64-es batch

In [ ]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

dataset

## A modell felépítése

A `tf.keras.Sequential` modell három rétege az Embedding, a GRU és a Dense. Az utolsó réteg időlépésenként a szókincs minden karakteréhez nyers pontszámot (logitot) rendel.

### T17 — A modell fő méretei

In [ ]:
# Length of the vocabulary in StringLookup Layer
vocab_size = len(vocab)

# The embedding dimension
embedding_dim = 256

# Number of RNN units
rnn_units = 1024

### T18 — Embedding, GRU, Dense

In [ ]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):

    model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim,batch_input_shape=[batch_size, None]),
    tf.keras.layers.GRU(rnn_units,return_sequences=True,stateful=True,recurrent_initializer='glorot_uniform'),
    tf.keras.layers.Dense(vocab_size)
  ])
    return model

#Ez a kaggleről van

### T19 — A tanítómodell létrehozása

In [ ]:
model = build_model(vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units, batch_size=BATCH_SIZE)

### T20 — A modell összefoglalója

In [ ]:
model.summary()

A GRU batchpozíciónként tárolja a belső állapotot, ezért a batchméret rögzített. A kevert sorozatok közötti állapotfolytonosság itt nem felel meg a könyv tényleges sorrendjének.

## A tanítatlan modell próbája

### T21 — Próba egy batchen

In [ ]:
for input_example_batch, target_example_batch in dataset.take(1):
  example_batch_predictions = model(input_example_batch)
  print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

### T22 — Második modell-összefoglaló

In [ ]:
model.summary()

A `tf.random.categorical` a logitokból mintáz. A nagyobb pontszámú lehetőségek gyakoribbak, de a legnagyobb pontszámú karakter nem minden futásban kerül kiválasztásra.

### T23 — Mintavétel a próbakimenetből

In [ ]:
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices,axis=-1).numpy()

### T24 — A mintavett karakterindexek

In [ ]:
sampled_indices

### T25 — A tanítatlan modell dekódolt próbája

In [ ]:
print("Input: \n", repr("".join(idx2char[input_example_batch[0]])))
print()
print("Next Char Predictions: \n", repr("".join(idx2char[sampled_indices ])))

## Tanítás és mentés

Adott időlépésben a helyes következő karakter osztályát kell jósolni.

### T26 — A veszteségfüggvény

In [ ]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)

### T27 — Kezdeti átlagos veszteség

In [ ]:
example_batch_mean_loss = loss(target_example_batch, example_batch_predictions)
print("Prediction shape: ", example_batch_predictions.shape, " # (batch_size, sequence_length, vocab_size)")
print("Mean loss:        ", example_batch_mean_loss)

Kezdetben egy közel egyenletes modellnél az átlagos veszteség exponenciálisa a karakterkészlet méretének közelében van.

### T28 — A veszteség exponenciálisa

In [ ]:
tf.exp(example_batch_mean_loss).numpy()

### T29 — A modell konfigurálása

In [ ]:
model.compile(optimizer='adam', loss=loss)

### Súlymentés

### T30 — Súlymentés beállítása

In [ ]:
# Directory where the checkpoints will be saved
checkpoint_dir = './training_checkpoints'
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True)

Az eredeti kód **30 epochát** futtat. A GPU-t használó futtatókörnyezetet hagyd csatlakoztatva, amíg befejeződik.

### T31 — Az epochák száma

In [ ]:
EPOCHS = 30

### T32 — A modell tanítása

In [ ]:
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

### T33 — A legutóbbi checkpoint

In [ ]:
tf.train.latest_checkpoint(checkpoint_dir)

Generáláshoz ugyanazokkal a rétegekkel új modellt építünk 1-es batchmérettel, majd betöltjük a helyi mentés súlyait.

### T34 — Egyes batch és visszatöltött súlyok

In [ ]:
model = build_model(vocab_size, embedding_dim, rnn_units, batch_size=1)

model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))

model.build(tf.TensorShape([1, None]))

## Szöveggenerálás és mintavétel

A kezdőszöveg után a függvény mintáz egy karaktert, és annak indexét használja a következő modellhívásban.

### T35 — A generáló függvény

In [ ]:
def generate_text(model, start_string):

    num_generate = 1000
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)


    text_generated = []

    temperature = 1.0

    model.reset_states()
    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0) #1-es dimenziókat eltávolítja
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)

        text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))

### T36 — Szöveg generálása az „Azok” prefixből

In [ ]:
print(generate_text(model, start_string=u"Azok"))

A maradék cellák egyetlen lépés alakjait mutatják meg. A tanított súlyok nem változnak, de a GRU belső állapota változhat.

### T37 — Egy generálási lépés külön

In [ ]:
text =[]
input_eval = [char2idx[s] for s in 'A']
input_eval = tf.expand_dims(input_eval, 0)
predictions = model(input_eval)

### T38 — Az egyes batchtengely eltávolítása

In [ ]:
predictions = tf.squeeze(predictions, 0)

### T39 — Egyetlen pontszám kiválasztása

In [ ]:
predictions[0,1]

### T40 — Egy karakterazonosító mintavétele

In [ ]:
predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

### T41 — Új bemenet és a karakter eltárolása

In [ ]:
input_eval = tf.expand_dims([predicted_id], 0)
text.append(idx2char[predicted_id])